use multiple records: 

In [1]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [2]:
# Selected MIT-BIH records
records = ['100', '101', '103', '105', '109']

# ECG sampling frequency
fs = 360

# Sliding window size
window_size = 5

# Store all processed samples
all_X = []
all_y = []

In [3]:
for record_id in records:

    print(f"Processing Record {record_id}...")

    # Load ECG signal
    record = wfdb.rdrecord(
        record_id,
        pn_dir='mitdb'
    )

    # Load annotations (R peaks)
    annotation = wfdb.rdann(
        record_id,
        'atr',
        pn_dir='mitdb'
    )

Processing Record 100...
Processing Record 101...
Processing Record 103...
Processing Record 105...
Processing Record 109...


extract RR intervals

In [4]:
# Extract R peak positions
r_peaks = annotation.sample

# Compute RR intervals
rr_intervals = np.diff(r_peaks) / fs

convert to HR

In [5]:
# Convert RR interval to Heart Rate
heart_rate = 60 / rr_intervals

create dataframe

In [6]:
df = pd.DataFrame({
    'heart_rate': heart_rate
})

print(df.head())

   heart_rate
0  232.258065
1   93.103448
2   94.736842
3   91.525424
4   89.626556


In [ ]:
outlier filtering

In [7]:
# Keep only physiological heart-rate range

df['heart_rate'] = df['heart_rate'].where(
    (df['heart_rate'] >= 40) &
    (df['heart_rate'] <= 180)
)

interpolation

In [8]:
# Fill missing values

df['heart_rate'] = df['heart_rate'].interpolate()

per record scaling

In [9]:
# Normalize EACH record independently

scaler = MinMaxScaler()

scaled_hr = scaler.fit_transform(
    df[['heart_rate']]
)

scaled_hr = scaled_hr.flatten()

sliding window

In [10]:
X_record = []
y_record = []

for i in range(len(scaled_hr) - window_size):

    # Previous values
    X_record.append(
        scaled_hr[i:i+window_size]
    )

    # Next value
    y_record.append(
        scaled_hr[i+window_size]
    )

X_record = np.array(X_record)
y_record = np.array(y_record)

store data

In [11]:
all_X.append(X_record)
all_y.append(y_record)

combine all records

In [12]:
X = np.concatenate(all_X)
y = np.concatenate(all_y)

print("Dataset Shape")
print(X.shape)
print(y.shape)

Dataset Shape
(2529, 5)
(2529,)


spilting train test

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    shuffle=False,
    random_state=42
)

SARIMA

random forest

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

XGBoost

LSTM